# Causal necessity and sufficiency (Section 3 and Appendix C)

**Paper section.** Section 3 (Causal necessity and sufficiency) with footnotes 2 and 3, and the estimated causal subspace of Appendix C.

**Claim.** In the paper's words, "negating the orthogonal component leaves the output unchanged at the three derived hooks (average KL ∼10^−11 nats at the logits, resid_final, and resid_post), whereas negating the causal component gives a substantially larger KL of 14.2, 13.5, and 13.5 nats, respectively", "At mlp_in, whose subspace is estimated rather than derived, negating the orthogonal component yields a small but nonzero KL of 0.041 nats (about 5% of the models' average per-token loss of 0.78 nats), compared with 2.1 nats for the causal component", and "the average KL between the target output and the model's output after the substitution is ∼10^−10 nats at both resid_final and resid_post."

**Experiment.** The negations at the logits, resid_final, and resid_post, the next-token loss, the two interventions at mlp_in with the estimated causal subspace, and the substitution of the causal component that produces a target output, for all 2,640 models, from `scripts/run_necessity_sufficiency.sh`.

**Saved Outputs.** None. The notebook prints the numbers of Section 3 and of Appendix C.

**How the notebook works.** The first code cell sets the paths and defines the loaders. Each following cell reads one experiment's files from `results/`, pools the free and the pinned models, and prints the averages that the paper reports.

In [ ]:
import glob, os, re, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ═══════════════════════════════════════════════════════════
# CONFIG: change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load(name, variant):
    """Every group of one experiment and one variant ('free' or 'pinned'), concatenated over models."""
    out = {}
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, name, variant, f'{name}_v*.npz'))):
        z = np.load(f, allow_pickle=True)
        for k in z.files:
            if z[k].ndim >= 1:
                out.setdefault(k, []).append(z[k])
    return {k: np.concatenate(v) for k, v in out.items()}

def load_all(name):
    """The free and the pinned models together."""
    a, b = load(name, 'free'), load(name, 'pinned')
    return {k: np.concatenate([a[k], b[k]]) for k in a if k in b}

def process_of(tag):
    """The process name of a model tag such as mess3_a85x05_4l_h4_s0."""
    return re.sub(r'_\d+l_h\d+_s\d+$', '', str(tag))

## Section 3: negation at the logits, resid_final, and resid_post

In [ ]:
neg = load_all('negation')
print(f"models: {len(neg['tags'])}")
for hook in ['logits', 'resid_final', 'resid_post']:
    causal, orth = neg[f'{hook}_causal_negate'], neg[f'{hook}_comp_negate']
    print(f"{hook:12s} causal component negated: mean KL {causal.mean():.1f} nats | "
          f"orthogonal component negated: mean KL {orth.mean():.1e} nats (largest magnitude {np.abs(orth).max():.1e})")

## Section 3 and Appendix C: the estimated causal subspace of mlp_in, its negation, and the loss

In [ ]:
mlp = load_all('mlp_in_negation')
loss = load_all('loss')['loss']
causal, orth = mlp['causal_negate_kl'].mean(), mlp['comp_matched_kl'].mean()
print(f"mlp_in, causal component negated: mean KL {causal:.1f} nats")
print(f"mlp_in, orthogonal component: mean KL {orth:.3f} nats")
print(f"average loss per token: {loss.mean():.2f} nats")
print(f"orthogonal KL as a share of the loss: {100 * orth / loss.mean():.0f}% | as a share of the causal KL: {100 * orth / causal:.0f}%")

# k99 of every model, and its median within each of the 22 processes
procs = np.array([process_of(t) for t in mlp['tags']])
k99_by_process = {p: np.median(mlp['k99'][procs == p]) for p in sorted(set(procs))}
medians = np.array(list(k99_by_process.values()))
print(f"k99, median per process: from {medians.min():g} to {medians.max():g}, median {np.median(medians):g}")
for p, k in k99_by_process.items():
    print(f"    {p:22s} {k:g}")

## Section 3: sufficiency at resid_final and resid_post

In [ ]:
steer = load_all('steering')
for key, hook in [('target_rf_kl', 'resid_final'), ('reach_post_kl', 'resid_post')]:
    print(f"{hook:12s} KL from the target to the output after the substitution: mean {steer[key].mean():.1e} nats "
          f"(largest magnitude {np.abs(steer[key]).max():.1e})")
print(f"share of token positions whose target was scaled down at resid_post (footnote 3): {100 * steer['reach_temp_frac'].mean():.0f}%")